Define the question

Mục tiêu của bài phân tích là phân tích dữ liệu bán hàng của cửa hàng cà phê để đánh giá tình hình kinh doanh và tìm ra các xu hướng quan trọng trong hoạt động bán hàng. Cụ thể, nhóm tập trung xem xét các yếu tố như doanh thu, số lượng sản phẩm bán ra, giá sản phẩm, loại sản phẩm, địa điểm cửa hàng và thời gian giao dịch, từ đó xác định những sản phẩm có mức tiêu thụ cao, sự thay đổi doanh thu theo thời gian và những đặc điểm nổi bật trong hành vi mua hàng. Đối tượng sử dụng kết quả phân tích là chủ cửa hàng, người quản lý và những người phụ trách hoạt động kinh doanh, nhằm hỗ trợ việc theo dõi hiệu quả bán hàng, quản lý sản phẩm và đưa ra các quyết định kinh doanh dựa trên dữ liệu. Tiêu chí đo lường thành công của bài phân tích là dữ liệu được tổng hợp đầy đủ và chính xác, các thông tin quan trọng được thể hiện bằng những chỉ số và biểu đồ phù hợp, đồng thời kết quả phân tích phải chỉ ra được các xu hướng, sản phẩm và thời điểm có ảnh hưởng đáng chú ý đến doanh thu, giúp người quản lý có cơ sở để đánh giá hoạt động kinh doanh và đề xuất hướng cải thiện trong tương lai.

In [ ]:
import os
import pandas as pd
import kagglehub

In [2]:
!pip install kagglehub

In [3]:
# Tải dataset Cafe Sales từ Kaggle
path1 = kagglehub.dataset_download(
    "ahmedmohamed2003/cafe-sales-dirty-data-for-cleaning-training"
)

In [4]:
# Tải dataset Cafe Sales từ Kaggle
path2 = kagglehub.dataset_download(
    "ahmedmohamedibrahim1/coffee-shop-sales-dataset"
)

In [5]:
# Kiểm tra các file có trong hai dataset
print(os.listdir(path1))
print(os.listdir(path2))

['dirty_cafe_sales.csv']
['Coffee Shop Sales.xlsx']


In [6]:
# Đọc dữ liệu từ hai nguồn

df1 = pd.read_csv(os.path.join(path1, "dirty_cafe_sales.csv"))

df2 = pd.read_excel(
    os.path.join(path2, "Coffee Shop Sales.xlsx"),
    sheet_name="Transactions"
)

print("Data 1:", df1.shape)
print("Data 2:", df2.shape)

Data 1: (10000, 8)
Data 2: (149116, 17)


In [7]:
# Kiểm tra tên các cột của hai dataset
print("Cột Data 1:")
print(df1.columns.tolist())

print("\nCột Data 2:")
print(df2.columns.tolist())

Cột Data 1:
['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent', 'Payment Method', 'Location', 'Transaction Date']

Cột Data 2:
['transaction_id', 'transaction_date', 'transaction_time', 'transaction_qty', 'store_id', 'store_location', 'product_id', 'unit_price', 'product_category', 'product_type', 'product_detail', 'Revenue', 'Month', 'Month.1', 'Weekday', 'Weekday.1', 'Hour']


In [8]:
#Đổi tên cột
df1 = df1.rename(columns={
    "Item": "Product Name",
    "Price Per Unit": "Price",
    "Location": "Store Location"
})

df2 = df2.rename(columns={
    "transaction_id": "Transaction ID",
    "transaction_date": "Transaction Date",
    "transaction_qty": "Quantity",
    "store_location": "Store Location",
    "product_detail": "Product Name",
    "unit_price": "Price"
})

In [9]:
#Kiểm tra
print(df1.columns.tolist())
print(df2.columns.tolist())

['Transaction ID', 'Product Name', 'Quantity', 'Price', 'Total Spent', 'Payment Method', 'Store Location', 'Transaction Date']
['Transaction ID', 'Transaction Date', 'transaction_time', 'Quantity', 'store_id', 'Store Location', 'product_id', 'Price', 'product_category', 'product_type', 'Product Name', 'Revenue', 'Month', 'Month.1', 'Weekday', 'Weekday.1', 'Hour']


In [10]:
# Chọn các cột chung giữa hai dataset để chuẩn bị concat
df1_common = df1[
    [
        "Transaction ID",
        "Transaction Date",
        "Product Name",
        "Quantity",
        "Price",
        "Store Location"
    ]
].copy()

df2_common = df2[
    [
        "Transaction ID",
        "Transaction Date",
        "Product Name",
        "Quantity",
        "Price",
        "Store Location"
    ]
].copy()

In [11]:
# Chuẩn hóa kiểu dữ liệu của mã giao dịch và ngày giao dịch
df1_common["Transaction ID"] = df1_common["Transaction ID"].astype(str)
df2_common["Transaction ID"] = df2_common["Transaction ID"].astype(str)

df1_common["Transaction Date"] = pd.to_datetime(
    df1_common["Transaction Date"],
    errors="coerce"
)

df2_common["Transaction Date"] = pd.to_datetime(
    df2_common["Transaction Date"],
    errors="coerce"
)

In [12]:
# Thêm cột Source để phân biệt dữ liệu đến từ nguồn nào
df1_common["Source"] = "Cafe Sales"
df2_common["Source"] = "Coffee Shop Sales"

In [13]:
# Gộp hai dataset theo chiều dọc bằng concat
df = pd.concat(
    [df1_common, df2_common],
    ignore_index=True
)

In [14]:
# Kiểm tra dataset sau khi gộp
print("Kích thước sau khi concat:", df.shape)

Kích thước sau khi concat: (159116, 7)


In [15]:
print(df.head())
print(df.tail())
print(df["Source"].value_counts())

  Transaction ID Transaction Date Product Name Quantity Price Store Location  \
0    TXN_1961373       2023-09-08       Coffee        2   2.0       Takeaway   
1    TXN_4977031       2023-05-16         Cake        4   3.0       In-store   
2    TXN_4271903       2023-07-19       Cookie        4   1.0       In-store   
3    TXN_7034554       2023-04-27        Salad        2   5.0        UNKNOWN   
4    TXN_3160411       2023-06-11       Coffee        2   2.0       In-store   

       Source  
0  Cafe Sales  
1  Cafe Sales  
2  Cafe Sales  
3  Cafe Sales  
4  Cafe Sales  
       Transaction ID Transaction Date          Product Name Quantity Price  \
159111         149452       2023-06-30         Peppermint Rg        2   2.5   
159112         149453       2023-06-30  English Breakfast Lg        2   3.0   
159113         149454       2023-06-30         Peppermint Lg        1   3.0   
159114         149455       2023-06-30            Cappuccino        1  3.75   
159115         149456       

In [16]:
# Kiểm tra tên cột, kiểu dữ liệu và giá trị thiếu
print("\nTên cột:")
print(df.columns.tolist())

print("\nKiểu dữ liệu:")
print(df.dtypes)

print("\nGiá trị thiếu:")
print(df.isnull().sum())


Tên cột:
['Transaction ID', 'Transaction Date', 'Product Name', 'Quantity', 'Price', 'Store Location', 'Source']

Kiểu dữ liệu:
Transaction ID              object
Transaction Date    datetime64[ns]
Product Name                object
Quantity                    object
Price                       object
Store Location              object
Source                      object
dtype: object

Giá trị thiếu:
Transaction ID         0
Transaction Date     460
Product Name         333
Quantity             138
Price                179
Store Location      3265
Source                 0
dtype: int64


In [17]:
# Chuyển Quantity và Price sang kiểu dữ liệu số
df["Quantity"] = pd.to_numeric(df["Quantity"], errors="coerce")
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

In [18]:
# Kiểm tra lại kiểu dữ liệu sau khi chuyển đổi
print(df.dtypes)

Transaction ID              object
Transaction Date    datetime64[ns]
Product Name                object
Quantity                   float64
Price                      float64
Store Location              object
Source                      object
dtype: object


In [19]:
# Kiểm tra kích thước và 5 dòng đầu của dataset cuối
print("Kích thước dataset:", df.shape)

print("\n5 dòng đầu:")
print(df.head())

Kích thước dataset: (159116, 7)

5 dòng đầu:
  Transaction ID Transaction Date Product Name  Quantity  Price  \
0    TXN_1961373       2023-09-08       Coffee       2.0    2.0   
1    TXN_4977031       2023-05-16         Cake       4.0    3.0   
2    TXN_4271903       2023-07-19       Cookie       4.0    1.0   
3    TXN_7034554       2023-04-27        Salad       2.0    5.0   
4    TXN_3160411       2023-06-11       Coffee       2.0    2.0   

  Store Location      Source  
0       Takeaway  Cafe Sales  
1       In-store  Cafe Sales  
2       In-store  Cafe Sales  
3        UNKNOWN  Cafe Sales  
4       In-store  Cafe Sales  


In [20]:
# Kiểm tra số lượng giá trị thiếu trong dataset cuối
print("\nSố lượng giá trị thiếu:")
print(df.isnull().sum())


Số lượng giá trị thiếu:
Transaction ID         0
Transaction Date     460
Product Name         333
Quantity             479
Price                533
Store Location      3265
Source                 0
dtype: int64


In [21]:
# Tạo thư mục data nếu thư mục chưa tồn tại

os.makedirs("data", exist_ok=True)

In [ ]:
# Xuất dataset đã gộp thành file CSV
df.to_csv(
    "data/coffee_shop_concat.csv",
    index=False
)

print("Đã xuất dataset chung.")

Đã xuất dataset chung.
